In [2]:
!pip install biopython pandas

   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   -------------------------------------- - 2.6/2.7 MB 20.1 MB/s eta 0:00:01
   ---------------------------------------- 2.7/2.7 MB 13.5 MB/s  0:00:00


In [5]:
from Bio import Entrez
import pandas as pd
import os

# --------------------------------------------------
# 0. SETTINGS
# --------------------------------------------------

# Put your actual email here
Entrez.email = "bhuvanavijayam19@gmail.com"

# Folder where you want to save the dataset
output_folder = r"E:\MAJOR PROJECT"

# Create the folder if it doesn't already exist
os.makedirs(output_folder, exist_ok=True)

# Final CSV path
output_path = os.path.join(output_folder, "mhealth_dataset.csv")


# --------------------------------------------------
# 1. PUBMED SEARCH QUERY
# --------------------------------------------------

query = '''
("mHealth" OR "mobile health")
AND applications
AND ("healthcare" OR "health monitoring" OR "health intervention")
AND 2020:2026[dp]
AND hasabstract[text]
'''

# Number of papers we want
MAX_RESULTS = 500


# --------------------------------------------------
# 2. SEARCH PUBMED
# --------------------------------------------------

print("Searching PubMed...")

search_handle = Entrez.esearch(
    db="pubmed",
    term=query,
    retmax=MAX_RESULTS,
    sort="relevance"
)

search_results = Entrez.read(search_handle)
search_handle.close()

pmids = search_results["IdList"]

print("Total papers found:", search_results["Count"])
print("PMIDs collected:", len(pmids))


# --------------------------------------------------
# 3. FETCH PAPER DETAILS
# --------------------------------------------------

print("\nFetching paper details...")

fetch_handle = Entrez.efetch(
    db="pubmed",
    id=pmids,
    retmode="xml"
)

records = Entrez.read(fetch_handle)
fetch_handle.close()


# --------------------------------------------------
# 4. EXTRACT INFORMATION
# --------------------------------------------------

papers = []

print("Processing papers...")

for article in records["PubmedArticle"]:

    citation = article["MedlineCitation"]
    article_info = citation["Article"]

    # --------------------------------------------------
    # PMID
    # --------------------------------------------------

    pmid = str(citation["PMID"])


    # --------------------------------------------------
    # TITLE
    # --------------------------------------------------

    title = str(article_info.get("ArticleTitle", ""))


    # --------------------------------------------------
    # ABSTRACT
    # --------------------------------------------------

    abstract_parts = []

    if "Abstract" in article_info:

        for part in article_info["Abstract"]["AbstractText"]:

            text = str(part)

            # Preserve section labels such as
            # BACKGROUND, METHODS, RESULTS, etc.
            if hasattr(part, "attributes"):

                label = part.attributes.get("Label")

                if label:
                    text = label + ": " + text

            abstract_parts.append(text)

    abstract = " ".join(abstract_parts)


    # --------------------------------------------------
    # YEAR
    # --------------------------------------------------

    year = ""

    if "Journal" in article_info:

        journal_info = article_info["Journal"]

        if "JournalIssue" in journal_info:

            if "PubDate" in journal_info["JournalIssue"]:

                pub_date = journal_info["JournalIssue"]["PubDate"]

                if "Year" in pub_date:
                    year = str(pub_date["Year"])


    # --------------------------------------------------
    # AUTHORS
    # --------------------------------------------------

    authors = []

    if "AuthorList" in article_info:

        for author in article_info["AuthorList"]:

            # Collective author / organization
            if "CollectiveName" in author:

                authors.append(
                    str(author["CollectiveName"])
                )

            else:

                last = author.get("LastName", "")
                first = author.get("ForeName", "")

                name = f"{first} {last}".strip()

                if name:
                    authors.append(name)

    authors = "; ".join(authors)


    # --------------------------------------------------
    # JOURNAL
    # --------------------------------------------------

    journal = ""

    if "Journal" in article_info:

        if "Title" in article_info["Journal"]:

            journal = str(
                article_info["Journal"]["Title"]
            )


    # --------------------------------------------------
    # DOI
    # --------------------------------------------------

    doi = ""

    if "PubmedData" in article:

        if "ArticleIdList" in article["PubmedData"]:

            for article_id in article["PubmedData"]["ArticleIdList"]:

                if article_id.attributes.get("IdType") == "doi":

                    doi = str(article_id)
                    break


    # --------------------------------------------------
    # STORE PAPER
    # --------------------------------------------------

    papers.append({

        "paper_id": len(papers) + 1,

        "pmid": pmid,

        "title": title,

        "abstract": abstract,

        "year": year,

        "authors": authors,

        "journal": journal,

        "doi": doi

    })


# --------------------------------------------------
# 5. CREATE DATAFRAME
# --------------------------------------------------

df = pd.DataFrame(papers)


# --------------------------------------------------
# 6. REMOVE PAPERS WITHOUT ABSTRACTS
# --------------------------------------------------

df = df[
    df["abstract"].str.strip() != ""
]


# --------------------------------------------------
# 7. RESET PAPER IDs
# --------------------------------------------------

df = df.reset_index(drop=True)

df["paper_id"] = range(
    1,
    len(df) + 1
)


# --------------------------------------------------
# 8. SAVE CSV TO YOUR PREFERRED LOCATION
# --------------------------------------------------

df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)


# --------------------------------------------------
# 9. DISPLAY RESULTS
# --------------------------------------------------

print("\n========================================")
print("DATASET CREATED SUCCESSFULLY!")
print("========================================")

print("\nNumber of papers:", len(df))

print("\nColumns:")
print(list(df.columns))

print("\nFirst 5 papers:")
print(
    df[
        [
            "paper_id",
            "pmid",
            "title",
            "year"
        ]
    ].head()
)

print("\nCSV saved at:")
print(output_path)

Searching PubMed...
Total papers found: 2187
PMIDs collected: 500

Fetching paper details...
Processing papers...

DATASET CREATED SUCCESSFULLY!

Number of papers: 500

Columns:
['paper_id', 'pmid', 'title', 'abstract', 'year', 'authors', 'journal', 'doi']

First 5 papers:
   paper_id      pmid                                              title  year
0         1  37171838  Problems and Barriers Related to the Use of Di...  2023
1         2  31678305  Mobile Health Technologies in Cardiopulmonary ...  2020
2         3  36118170  mHealth and telemedicine utility in the monito...  2022
3         4  31825132  Mobile health and cardiac rehabilitation in ol...  2020
4         5  37126398  Applications of Federated Learning in Mobile H...  2023

CSV saved at:
E:\MAJOR PROJECT\mhealth_dataset.csv


In [2]:
from Bio import Entrez, Medline
import pandas as pd
import time

# -------------------------------------------------
# 1. CONFIGURATION
# -------------------------------------------------

# Replace with your email
Entrez.email = "your_email@gmail.com"

# Optional: Add your NCBI API key here
# Entrez.api_key = "YOUR_NCBI_API_KEY"

SEARCH_QUERY = (
    '("mHealth"[Title/Abstract] OR '
    '"mobile health"[Title/Abstract] OR '
    '"mobile health applications"[Title/Abstract] OR '
    '"wearable health"[Title/Abstract] OR '
    '"remote patient monitoring"[Title/Abstract])'
)

MAX_RESULTS = 500


# -------------------------------------------------
# 2. SEARCH PUBMED AND GET PMIDs
# -------------------------------------------------

print("Searching PubMed...")

handle = Entrez.esearch(
    db="pubmed",
    term=SEARCH_QUERY,
    retmax=MAX_RESULTS
)

search_results = Entrez.read(handle)
handle.close()

pmids = search_results["IdList"]

print(f"Found {len(pmids)} papers.")


# -------------------------------------------------
# 3. FETCH PAPER DETAILS
# -------------------------------------------------

print("Fetching paper details...")

records = []

# Fetch in batches
BATCH_SIZE = 100

for start in range(0, len(pmids), BATCH_SIZE):

    batch = pmids[start:start + BATCH_SIZE]

    handle = Entrez.efetch(
        db="pubmed",
        id=",".join(batch),
        rettype="medline",
        retmode="text"
    )

    batch_records = Medline.parse(handle)

    for record in batch_records:

        pmid = record.get("PMID", "")
        title = record.get("TI", "")
        abstract = record.get("AB", "")

        # Publication year
        year = record.get("DP", "")

        # MeSH terms
        mesh_terms = record.get("MH", [])
        mesh_terms = "; ".join(mesh_terms)

        records.append({
            "pmid": pmid,
            "title": title,
            "abstract": abstract,
            "year": year,
            "mesh_terms": mesh_terms
        })

    handle.close()

    print(f"Processed {min(start + BATCH_SIZE, len(pmids))} papers")

    time.sleep(0.4)


# -------------------------------------------------
# 4. CREATE DATAFRAME
# -------------------------------------------------

df = pd.DataFrame(records)

print("\nBefore cleaning:", len(df))


# -------------------------------------------------
# 5. REMOVE DUPLICATES
# -------------------------------------------------

df.drop_duplicates(
    subset=["pmid"],
    inplace=True
)


# -------------------------------------------------
# 6. REMOVE MISSING ABSTRACTS
# -------------------------------------------------

df["abstract"] = df["abstract"].fillna("").str.strip()
df["title"] = df["title"].fillna("").str.strip()

df = df[
    (df["abstract"] != "") &
    (df["title"] != "")
]


print("After cleaning:", len(df))


# -------------------------------------------------
# 7. SAVE AS CSV
# -------------------------------------------------

OUTPUT_PATH = r"E:\MAJOR PROJECT\mhealth_pubmed_dataset.csv"

df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8"
)

print("\nDataset saved successfully!")
print(f"File saved at: {OUTPUT_PATH}")

print("\nSample data:")
print(df.head())

Searching PubMed...
Found 500 papers.
Fetching paper details...
Processed 100 papers
Processed 200 papers
Processed 300 papers
Processed 400 papers
Processed 500 papers

Before cleaning: 500
After cleaning: 491

Dataset saved successfully!
File saved at: E:\MAJOR PROJECT\mhealth_pubmed_dataset.csv

Sample data:
       pmid                                              title  \
0  42666381  A user centered evaluation framework for mobil...   
1  42666156  Scale-up of the Diactive-1 mHealth program int...   
2  42663398  Telehealth and maternal referral services: A s...   
3  42662113  Age and the digital divide in the use of healt...   
4  42662000  How digital health tools shape emotional suppo...   

                                            abstract          year  \
0  BACKGROUND: The rapid growth of digital mental...          2026   
1  INTRODUCTION: Physical activity is strongly re...          2026   
2  BackgroundHigh-quality healthcare during pregn...  2026 Jan-Dec   
3  INTRODU